In [48]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import os
import scipy

# charger la data processed
data_filled = pd.read_csv('../data/processed/chip_chain_processed.csv')
data_daily_returns = pd.read_csv('../data/processed/chip_chain_daily_returns.csv')
data_log_returns = pd.read_csv('../data/processed/chip_chain_log_returns.csv')
data_indicators = pd.read_csv('../data/processed/chip_chain_custom_indicators.csv')
data_state_log = pd.read_csv('../data/processed/chip_chain_market_regimes.csv')
data_state_simple_returns = pd.read_csv('../data/processed/chip_chain_market_regimes_simple_returns.csv')

Avant on s'occupait seulement de trouver les différents états et structure du marché, mais maintenant on souhaite transformer la vision probabilistique du marché en allocation de capital (donc plus concret) avec Markowitz

L'idée derrière Markowitz, c'est que si l'on a 1 millions d'euros à investir, on pourrait :
- tout mettre sur NVDA car énorme rendement, en explosion, mais beaucoup de risque
- tout mettre sur une action assez stable comme AAPL, mais moins de rendement
- ou bien de diversifier avec plusieurs actifs

Evidemment la bonne solution pour être plus sur de tout perdre et de faire de bon gain est de diversifier ses actifs (voire hors tech pour être plus sûr mais ici ce n'est pas le but (mais on a des valeurs énergétiques pour faire un peu cela)). Ainsi, la question n'est pas qu'elle est le meilleur actif, mais de trouver la meilleur combinaison d'actifs avec la meilleur combinaison rendement/risque. C'est comme ça qu'on est arrivéau portfolio et la théorie qui va avec avec Harry Markowitz, on ne regarde plus actif par actif mais aussi leur corrélation, covariance

On calcul donc le rendement espéré du portefeuille comme suit :
$$ R_p=w^T\mu \qquad avec\ le\ vecteur\ de\ poids\ :\ w = \begin{bmatrix} w_1 \\ w_2 \\ . \\ . \\ . \\ w_n \end{bmatrix}\quad où\ \sum_{i=1}^nw_i=1\qquad et\ le\ vecteur\ de\ rendement\ :\ \mu = \begin{bmatrix} \mu_1 \\ \mu_2 \\ . \\ . \\ . \\ \mu_n \end{bmatrix} $$

Exemple on estime que :
- NVDA rapporte en moyenne 20 % et on met 50% du capital
- AAPL 10% et on met 30%
- USO 5% et on met 20%

alors : $$ R_p = 0.5*0.2+0.3*0.1+0.2*0.05 = 0.1+0.03+0.01 = 0.14 $$

Alors là ça parait peut être simple mais cela repose beaucoup sur le $\mu$ choisi car comment le calcule-t-on, sur les rendements passés ?, sur une moyenne de combien de jours ? (1 semaine, 2 semaines, 1 mois, 1 trimestre, 1 an), cela peut grandement faire varier le $\mu$ donc $R_p$

De plus, les actifs interagissent entre eux et c'est sur cette partie qu'agit Markowitz (dommage pour le $\mu$ on va devoir trouvé par nous même), il utilise la covariance pour si A monte est ce que B monte aussi :
- covariance positive donc ils bougent ensemble
- covariance négatives donc ils se compensent
- covariance nulle donc ils sont indépendants

Ainsi à partir de la matrice de covariance $ \Sigma =  \begin{pmatrix} \sigma_A^2 & Cov(A,B) \\ Cov(B,A) & \sigma_B^2 \end{pmatrix}$, on calcul le risque du portefeuille :
$$ \sigma_p = \sqrt{w^T\Sigma w} $$
Si on regarde bien $w^T\Sigma w$ additione les variances individuelles mais aussi toutes les covariances croisées. Ainsi :
- si les actifs sont corrélés, alors le risque explose
- si les actifs se compensent, alors le risque diminue

Le Markowitz classique suppose $\mu$ et $\Sigma$ constants mais le marché n'est pas stationnaire :
- les corrélations changent
- la volatilité change
- les états changent

Donc $\Sigma_{bull} \ne \Sigma_{bear}$ et $\mu_{bull} \ne \mu_{bear}$

Ainsi l'idée est d'avoir un Markowitz adaptatif avec $\Sigma^{(0)}, \Sigma^{(1)}, \Sigma^{(2)}$ et $\mu^{(0)}, \mu^{(1)}, \mu^{(2)}$, donc :
- en marché bull, un portefeuille agressif
- en marché bear, un portefeuille défensif
- en marché neutre, un compromis

Néanmoins, le rendement seul ne suffit pas, s'il on a :
- portefeuille A
    - rendement 30%
    - volatilité 80%
- portefeuille B
    - rendement 15%
    - volatilité 10%

Lequel est le meilleur ? C'est pourquoi on utilise le ratio de Sharpe pour mesurer combien de rendement on obtient par unité de risque donc :
$$ Sharpe = \frac{R_p-R_f}{\sigma_p}\quad avec\ R_f = taux\ sans\ risque $$

Evidemment on ne veut pas calculer toutes les combinaisons pour trouver la meilleur, on veut $max_wSharpe(w)$ mais il n'existe pas de max que des min car c'est dans la plpart des cas on veut minimiser, c'est pourquoi on utilise évidemment $ max_wSharpe(w) = -min_w-Sharpe(w) $, ainsi on doit calculer $-Sharpe(w)$

In [49]:
def portfolio_performance(w, mean_returns, cov_matrix):
    returns = np.sum(mean_returns * w) * 252
    std = np.sqrt(np.dot(w.T, np.dot(cov_matrix, w))) * np.sqrt(252)
    return returns, std

def negative_sharpe_ratio(w, mean_returns, cov_matrix, risk_free_rate=0.02):
    returns, std = portfolio_performance(w, mean_returns, cov_matrix)
    sharpe_ratio = (returns - risk_free_rate) / std
    return -sharpe_ratio

def optimize_portfolio(mean_returns, cov_matrix, num_assets, risk_free_rate=0.02):
    args = (mean_returns, cov_matrix, risk_free_rate)
    constraints = ({'type': 'eq', 'fun': lambda x: np.sum(x) - 1})
    bounds = tuple((0, 1) for _ in range(num_assets))
    initial_guess = num_assets * [1. / num_assets]
    
    result = scipy.optimize.minimize(negative_sharpe_ratio, initial_guess, args=args,
                      method='SLSQP', bounds=bounds, constraints=constraints)
    
    return result.x

On utilise scipy pour trouver le min de -Sharpe qui est un algorithùe d'optimisation non-linéaire SLSQP (Sequential Least Squares Programming), il a besoin de :
- la fonction cible (negative_sharpe_ratio)
- les contraintes (la somme des poids est égale à 1 (100% du capital investi))
- les bornes (chaque poids est compris entre 0 et 1 (on s'inderdit la vente à découvert))
- (une matrice initial des poids)

On fait cela sur un petit groupe de manière classique (Markowitz classique sans prendre en compte les états). On ne traville pas sur les rendements log pour markowitz classique car ils sont additifs dans le temps mais le log d'une somme n'est pas égal à la somme des logs. C'est pour cela que l'on travaille sur les rendements simples qui sont additifs dans l'espace

In [50]:
small_portfolio = data_daily_returns[['USO', 'ASML', 'TSM', 'NVDA', 'AAPL', 'MSFT']]
mean_returns = small_portfolio.mean()
cov_matrix = small_portfolio.cov()
num_assets = len(small_portfolio.columns)
optimal_weights = optimize_portfolio(mean_returns, cov_matrix, num_assets)
print("Répartition des poids en prenant en compte toute la période:")
print("Optimal Weights:", optimal_weights)
optimal_weights_rounded = np.round(optimal_weights, 4)
print("Optimal Weights (rounded):", optimal_weights_rounded)

small_portfolio_1year = small_portfolio.tail(252) # tail permet de prendre les données de la dernière année
mean_returns = small_portfolio_1year.mean()
cov_matrix = small_portfolio_1year.cov()
num_assets = len(small_portfolio_1year.columns)
optimal_weights_1year = optimize_portfolio(mean_returns, cov_matrix, num_assets)
print("Répartition des poids en prenant en compte la dernière année:")
print("Optimal Weights (1 year):", optimal_weights_1year)
optimal_weights_rounded_1year = np.round(optimal_weights_1year, 4)
print("Optimal Weights (1 year, rounded):", optimal_weights_rounded_1year)

Répartition des poids en prenant en compte toute la période:
Optimal Weights: [4.38017678e-17 0.00000000e+00 1.59319315e-01 4.75439102e-01
 1.95872785e-01 1.69368798e-01]
Optimal Weights (rounded): [0.     0.     0.1593 0.4754 0.1959 0.1694]
Répartition des poids en prenant en compte la dernière année:
Optimal Weights (1 year): [0.         0.56338799 0.43661201 0.         0.         0.        ]
Optimal Weights (1 year, rounded): [0.     0.5634 0.4366 0.     0.     0.    ]


On voit bien qu'en fonction de ce que l'on prend en donnée (la dernière année ou toute la donnée), les données sont complètement différentes, c'est pourquoi on va voie ce que ça donne avec un modèle adpatatif pour Markowitz en prenant en compte les états comme vu en théorie précédemment

In [51]:
# on peut faire la même chose mais en prenant en compte les régimes de marché avec toujours les rendements simples
small_adaptive_portfolio = data_state_simple_returns[['USO', 'ASML', 'TSM', 'NVDA', 'AAPL', 'MSFT']]
small_adaptive_portfolio['Market_Regime'] = data_state_simple_returns['Market_Regime_General']
mean_returns = small_adaptive_portfolio.groupby('Market_Regime').mean()
print("Mean Returns by Market Regime:")
print(mean_returns)
cov_matrix = small_adaptive_portfolio.groupby('Market_Regime').cov()
print("Covariance Matrix by Market Regime:")
print(cov_matrix)
num_assets = len(small_adaptive_portfolio.columns) - 1
optimal_weights_adaptive = {}

Mean Returns by Market Regime:
                    USO      ASML       TSM      NVDA      AAPL      MSFT
Market_Regime                                                            
0             -0.001637  0.000217  0.000375  0.001216  0.000095  0.000061
1              0.000816  0.002066  0.001586  0.002808  0.001274  0.001229
2             -0.000404  0.000765  0.001119  0.002287  0.001052  0.001192
Covariance Matrix by Market Regime:
                         USO      ASML       TSM      NVDA      AAPL      MSFT
Market_Regime                                                                 
0             USO   0.001280  0.000291  0.000249  0.000279  0.000251  0.000214
              ASML  0.000291  0.001119  0.000741  0.001061  0.000557  0.000549
              TSM   0.000249  0.000741  0.000885  0.000948  0.000458  0.000446
              NVDA  0.000279  0.001061  0.000948  0.001827  0.000744  0.000761
              AAPL  0.000251  0.000557  0.000458  0.000744  0.000704  0.000518
          

In [52]:
# si 0 mettre bear market, si 1 mettre crab market, si 2 mettre bull market
regime_names = {0: "Bear Market", 1: "Crab Market", 2: "Bull Market"}
for regime in mean_returns.index:
    optimal_weights_adaptive[regime] = optimize_portfolio(mean_returns.loc[regime], cov_matrix.loc[regime], num_assets)
    print(f"Optimal Weights for {regime_names[regime]}:", optimal_weights_adaptive[regime])
    optimal_weights_rounded_adaptive = np.round(optimal_weights_adaptive[regime], 4)
    print(f"Optimal Weights for {regime_names[regime]} (rounded):", optimal_weights_rounded_adaptive)

Optimal Weights for Bear Market: [1.24426417e-17 0.00000000e+00 4.16333634e-17 1.00000000e+00
 0.00000000e+00 0.00000000e+00]
Optimal Weights for Bear Market (rounded): [0. 0. 0. 1. 0. 0.]
Optimal Weights for Crab Market: [0.12634195 0.29055901 0.07669857 0.30932856 0.14438305 0.05268885]
Optimal Weights for Crab Market (rounded): [0.1263 0.2906 0.0767 0.3093 0.1444 0.0527]
Optimal Weights for Bull Market: [6.76000055e-17 1.44740990e-16 1.49084317e-01 3.21521853e-01
 1.94168276e-01 3.35225553e-01]
Optimal Weights for Bull Market (rounded): [0.     0.     0.1491 0.3215 0.1942 0.3352]


In [53]:
small_adaptive_portfolio.groupby('Market_Regime').std()

,USO,ASML,TSM,NVDA,AAPL,MSFT
Market_Regime,,,,,,
0,0.035773,0.033453,0.029756,0.042740,0.026538,0.024466
1,0.019748,0.021781,0.019410,0.028199,0.017245,0.015739
2,0.017686,0.015824,0.013935,0.020210,0.013312,0.012166


## Bear Market

| Actif | Volatilité |
|---|---|
| NVDA | 4.27% |
| USO | 3.57% |
| ASML | 3.35% |

La volatilité explose fortement.

Cela traduit :
- stress de marché,
- incertitude,
- mouvements brutaux.


## Bull Market

| Actif | Volatilité |
|---|---|
| NVDA | 2.02% |
| AAPL | 1.33% |
| MSFT | 1.22% |

La volatilité devient beaucoup plus faible.

Le HMM identifie donc un régime de compression de volatilité.


In [54]:
small_adaptive_portfolio.groupby('Market_Regime').corr()

USO      ASML       TSM      NVDA      AAPL      MSFT
Market_Regime                                                                 
0             USO   1.000000  0.243520  0.233868  0.182543  0.264893  0.244529
              ASML  0.243520  1.000000  0.744508  0.742089  0.627693  0.670459
              TSM   0.233868  0.744508  1.000000  0.745395  0.579840  0.613221
              NVDA  0.182543  0.742089  0.745395  1.000000  0.655714  0.727551
              AAPL  0.264893  0.627693  0.579840  0.655714  1.000000  0.798543
              MSFT  0.244529  0.670459  0.613221  0.727551  0.798543  1.000000
1             USO   1.000000  0.174888  0.180039  0.147376  0.181297  0.194519
              ASML  0.174888  1.000000  0.615434  0.558832  0.455154  0.507627
              TSM   0.180039  0.615434  1.000000  0.540949  0.428825  0.465552
              NVDA  0.147376  0.558832  0.540949  1.000000  0.435904  0.543466
              AAPL  0.181297  0.455154  0.428825  0.435904  1.000000  0.534002
              MSFT  0.194519  0.507627  0.465552  0.543466  0.534002  1.000000
2             USO   1.000000  0.079677  0.084622  0.079858  0.070607  0.102414
              ASML  0.079677  1.000000  0.455734  0.405334  0.320555  0.369346
              TSM   0.084622  0.455734  1.000000  0.374071  0.291263  0.300268
              NVDA  0.079858  0.405334  0.374071  1.000000  0.278149  0.368860
              AAPL  0.070607  0.320555  0.291263  0.278149  1.000000  0.332193
              MSFT  0.102414  0.369346  0.300268  0.368860  0.332193  1.000000

## Bear Market : explosion des corrélations

Exemples :

| Paires | Corrélation |
|---|---|
| ASML-NVDA | 0.742 |
| TSM-NVDA | 0.745 |
| AAPL-MSFT | 0.799 |

Les actifs technologiques deviennent fortement corrélés.

### Pourquoi cela arrive ?

En période de stress :
- les investisseurs vendent les secteurs entiers,
- les fondamentaux individuels deviennent secondaires,
- les flux macro dominent.

Les actions technologiques deviennent alors un unique facteur de risque.


### Conséquence sur Markowitz

Le risque du portefeuille dépend des covariances :

$$
\sigma _p^2 = w^T \Sigma w
$$

Quand les corrélations augmentent :
- les termes de covariance explosent,
- la diversification devient inefficace.

Le portefeuille devient donc beaucoup plus risqué.

### Bull Market : retour de la diversification

Dans le Bull Market :

| Paires | Corrélation |
|---|---|
| NVDA-AAPL | 0.278 |
| NVDA-MSFT | 0.369 |

Les corrélations deviennent beaucoup plus faibles.

Cela signifie que :
- les entreprises suivent davantage leurs dynamiques propres,
- la diversification redevient efficace.

In [60]:
# calcul du sharpe ratio par actif et par régime de marché à annualiser
risk_free_rate = 0.02
sharpe_ratios = {}
for regime in mean_returns.index:
    returns = mean_returns.loc[regime] * 252    # on annualise les rendements
    std = np.sqrt(np.diag(cov_matrix.loc[regime])) * np.sqrt(252)  # on annualise les écarts-types
    sharpe_ratios[regime] = (returns - risk_free_rate) / std
print("Sharpe Ratios by Market Regime:")
for regime in sharpe_ratios:
    print(f"{regime_names[regime]}: {sharpe_ratios[regime]}")

Sharpe Ratios by Market Regime:
Bear Market: USO    -0.761696
ASML    0.065325
TSM     0.157532
NVDA    0.422356
AAPL    0.009221
MSFT   -0.011857
Name: 0, dtype: float64
Crab Market: USO     0.592236
ASML    1.448249
TSM     1.232180
NVDA    1.536217
AAPL    1.099464
MSFT    1.159417
Name: 1, dtype: float64
Bull Market: USO    -0.433631
ASML    0.687414
TSM     1.184288
NVDA    1.733953
AAPL    1.159430
MSFT    1.452170
Name: 2, dtype: float64


# Analyse des Sharpes par régime

## Bear Market

| Actif | Sharpe |
|---|---|
| NVDA | 0.42 |
| TSM | 0.16 |
| ASML | 0.07 |

NVDA possède le meilleur ratio rendement/risque malgré sa forte volatilité.

Cela explique pourquoi l’optimiseur alloue 100% du portefeuille à NVDA.


## Pourquoi ce résultat n’est pas absurde

Même en régime de stress :
- NVDA conserve un rendement moyen élevé,
- suffisamment important pour compenser son risque.

Markowitz optimise :
- le rendement ajusté du risque,
- pas simplement la faible volatilité.

## Crab Market

| Actif | Sharpe |
|---|---|
| NVDA | 1.54 |
| ASML | 1.45 |
| TSM | 1.23 |

Le régime Crab offre :
- volatilité modérée,
- corrélations intermédiaires,
- rendements encore élevés.

La diversification fonctionne donc particulièrement bien.

Le portefeuille devient beaucoup plus équilibré.

## Bull Market

| Actif | Sharpe |
|---|---|
| NVDA | 1.73 |
| MSFT | 1.45 |
| TSM | 1.18 |

Les grandes valeurs technologiques dominent fortement le régime haussier.

# Pourquoi certains actifs disparaissent

Par exemple ASML en Bull Market.

Même si ASML possède un bon rendement :
- NVDA possède un Sharpe supérieur,
- avec une covariance similaire.

L’optimiseur considère donc qu’ASML apporte peu de diversification marginale supplémentaire.

Markowitz ne sélectionne pas les meilleurs actifs individuellementm mais les actifs apportant le meilleur compromis rendement / covariance.

# Résultat principal de l’étude

Cette étude montre que :

## Les régimes modifient fortement :
- les volatilités,
- les corrélations,
- les ratios de Sharpe,
- les allocations optimales.

# Conclusion financière

Le modèle de Markowitz classique suppose implicitement que $\mu$ et $\Sigma$ sont constants dans le temps.

Or les résultats montrent clairement que cette hypothèse est fausse.

Les marchés possèdent des régimes distincts avec :
- des structures de risque différentes,
- des structures de corrélation différentes,
- des portefeuilles optimaux différents.

Le Markowitz adaptatif conditionné par HMM permet donc :
- d’adapter dynamiquement l’allocation,
- de mieux prendre en compte la non-stationnarité des marchés,
- et d’obtenir une gestion du risque plus réaliste.

# Nuances et limites
## NVDA en crise
Le modèle met 100% en crise sur NVDA ce qui est du au bon résultat même en temps de crise certes mais surtout qu'il a rebondi plus tôt que les autres ainsi souvent lorsque NVDA était déjà reparti à la hausse, les autres actions étaient encore en situation de crise et puisque l'on prend en compte un état général on perd cette info.

## Problème de diversification
Certaines fois notamment pour la crise on est trop focus sur une donnée pourquoi pas, on pourrait se dire mais dans ce cas ci surtout si c'est une donnée très volatile on doit être plus diversifié et surtout lorsque l'on voit que l'impact du $\mu$ est très importante (une petite différence peut faire que l'on met tout sur un seul actif) donc on dévrait plutôt mettre un poids maximal (30% par exemple), de plus il serait intéressant d'avoir des actifs inversement corrélés pour vraiment voir ce changement comme par exemple avec l'or en temps de crise

## Problème avec le temps réel
Pour trouver les états, on utilise l'algorithme de Viterbi qui se régarde toutes la séquence passé et futur pour trouver le meilleur chemin global, mais le jour où la crise éclate, on a pas les données de demain et donc ça prend au moins de 3, 5 jours à comprendre que l'on est plus dans un bull market et donc on prend tout les effets du krach en pleine figure, ce serait mieux de prendre l'algorithme forward qui ne prend qu'en compte le passé

## Les crises sont différentes
Prenont de crise pour montrer que cela n'a pas du tout le même impact :
- la crise Covid, tout le monde est confiné, le télétravaille augmente donc la demande en outil numérique aussi, ainsi la tech explose
- si la Chine envahissait Taïwan, la chaine de production est rompu du moins pour l'Occident ou au moins affaibli (et encore cela est seulement si la Chine arrive à ce que les usines ne soient pas détruites notamment celles de TSMC) et donc la tech s'effondrent

## On ne peut pas miser sur la baisse
On a fixer comme quoi on ne peut pas misser sur la baisse en bornant les poids entre 0 et 1 donc on ne peut pas miser sur une baisse et donc vendre ce qui est souvent à faire au moins sur une parti en crise (à nuancer)